# qdmpy Tutorial: ODMR Data Analysis

This tutorial walks through a complete QDM analysis workflow:

1. **Load** ODMR data from MATLAB files using `MatlabLoader` and `ODMRData`
2. **Process** data with a modular processor pipeline (binning, normalization, fluorescence correction)
3. **Fit** spectra with GPU-accelerated `FitManager` (ESR14N model)
4. **Visualise** B₁₁₁ remanent and induced field maps from `FitResult`

**Dataset**: `MIL2_FOV1` — two MATLAB files (neg/pos field polarity), two frequency ranges
(low 2.72–2.87 GHz, high 2.87–3.02 GHz), 51 frequency steps, 1200 × 1920 pixels.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from qdmpy.odmr import ODMRData, ODMR
from qdmpy.odmr.io import MatlabLoader
from qdmpy.odmr.processors import (
    BinningProcessor,
    FluorescenceCorrectionProcessor,
    NormalizationProcessor,
)
from qdmpy.fitting import FitManager

## 1. Load ODMR data

`MatlabLoader` reads `run_*.mat` files from the data folder.  
`ODMRData.from_loader()` wraps the result in a validated 5-D `xr.DataArray`
with named dimensions `(polarity, freq_range, y, x, freq_idx)`.

In [ ]:
DATA_FOLDER = Path.home() / "git" / "qdmpy" / "tests" / "data" / "MIL2_FOV1"

loader = MatlabLoader(data_folder=str(DATA_FOLDER))
odmr_data = ODMRData.from_loader(loader)

print(odmr_data.data)

In [ ]:
# Frequency grid: shape (n_frange, n_freq)
freq_ghz = odmr_data.data.coords["freq_ghz"].values
franges = list(odmr_data.data.coords["freq_range"].values)

print("Frequency ranges:")
for label, freqs in zip(franges, freq_ghz):
    step = (freqs[-1] - freqs[0]) / (len(freqs) - 1) * 1000
    print(f"  {label}: {freqs[0]:.4f} – {freqs[-1]:.4f} GHz  "
          f"({len(freqs)} steps, Δf ≈ {step:.2f} MHz)")

## 2. Visualise mean spectra

Before processing, inspect the spatially-averaged ODMR spectra. Each panel shows one
frequency range; the two curves correspond to the negative and positive field polarities.

In [ ]:
mean_spectra = odmr_data.data.mean(dim=["y", "x"])
pols = list(odmr_data.data.coords["polarity"].values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for i_frange, frange in enumerate(franges):
    ax = axes[i_frange]
    freqs = freq_ghz[i_frange]
    for pol in pols:
        spectrum = mean_spectra.sel(polarity=pol, freq_range=frange).values
        ax.plot(freqs, spectrum, "o-", markersize=3, label=pol)
    ax.set_xlabel("Frequency (GHz)")
    ax.set_ylabel("Fluorescence (counts)")
    ax.set_title(f"Mean spectrum — {frange} range")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle("Raw ODMR spectra (spatially averaged)", y=1.01)
plt.tight_layout()
plt.show()

## 3. Processing pipeline

The `ODMR` class manages a **processor pipeline** applied sequentially to the raw data:

| Processor | Purpose |
|-----------|--------|
| `BinningProcessor(bin_factor=4)` | Spatial 4×4 averaging — reduces noise, 1920×1200 → 480×300 |
| `NormalizationProcessor(method="max")` | Divide by per-pixel max so dip depth = contrast |
| `FluorescenceCorrectionProcessor()` | Remove spatially varying fluorescence background |

Each processor returns a new `ODMRData`; the original data is never mutated.

In [ ]:
odmr = ODMR(odmr_data)

odmr.processor_manager.add_processor(BinningProcessor(bin_factor=4))
odmr.processor_manager.add_processor(NormalizationProcessor(method="max"))
odmr.processor_manager.add_processor(FluorescenceCorrectionProcessor())
odmr.process_data()

proc = odmr.processed_data
print(f"Raw shape:       {odmr_data.data.shape}")
print(f"Processed shape: {proc.data.shape}")

### Inspect a processed spectrum

After binning and normalization the Lorentzian dips are cleaner and the baseline is at 1.

In [ ]:
freq_ghz_proc = proc.data.coords["freq_ghz"].values
pols_proc = list(proc.data.coords["polarity"].values)
franges_proc = list(proc.data.coords["freq_range"].values)

# Centre pixel of the binned image
cy, cx = proc.data.sizes["y"] // 2, proc.data.sizes["x"] // 2

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for i_pol, pol in enumerate(pols_proc):
    for i_frange, frange in enumerate(franges_proc):
        ax = axes[i_pol, i_frange]
        freqs = freq_ghz_proc[i_frange]
        spectrum = proc.data.sel(polarity=pol, freq_range=frange).isel(y=cy, x=cx).values
        ax.plot(freqs, spectrum, "o-", markersize=3, color="C0")
        ax.set_xlabel("Frequency (GHz)")
        ax.set_ylabel("Normalised intensity")
        ax.set_title(f"pol={pol}, range={frange} | pixel ({cy}, {cx})")
        ax.grid(True, alpha=0.3)

plt.suptitle("Processed ODMR spectra (4×4 binned, normalised)", y=1.01)
plt.tight_layout()
plt.show()

## 4. Spectral fitting with FitManager

`FitManager` is stateless: construct it once with a model name, call `.fit()` with
data and frequencies, receive a `FitResult`.  The same manager can be reused on
different datasets.

Model `"ESR14N"` fits three Lorentzian dips with ¹⁴N hyperfine splitting (Ahyp = 2.16 MHz).
Parameters per pixel per (pol, frange): `center`, `width`, `contrast_0/1/2`, `offset`, `chi2`, `states`.

In [ ]:
proc_da = proc.data  # xr.DataArray: (polarity, freq_range, y, x, freq_idx)

fitm = FitManager("ESR14N")
res = fitm.fit(proc_da, freq_ghz_proc)  # freq_ghz_proc has shape (n_frange, n_freq)

metrics = res.get_fit_quality_metrics()
print(res)
print(f"\nConvergence rate: {metrics['convergence_rate']:.3%}")
print(f"Mean chi²:        {metrics['mean_chi2']:.2e}")
print(f"Total fit time:   {metrics['total_fit_time']:.1f} s")

## 5. B₁₁₁ magnetic field maps

`FitResult` computes B₁₁₁ automatically from the fitted resonance centres:

```
δB[pol] = sign[pol] × (f_high − f_low) / 2 / γ_NV     [µT]
b111_remanent = (δB_neg + δB_pos) / 2    # permanent magnetisation
b111_induced  = (δB_neg − δB_pos) / 2    # paramagnetic / bias-tracking
```

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

vmax = np.percentile(np.abs(res.b111_remanent), 99)
im0 = axes[0].imshow(res.b111_remanent, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0].set_title("B\u2081\u2081\u2081 remanent (\u00b5T)")
axes[0].set_xlabel("x (pixels)")
axes[0].set_ylabel("y (pixels)")
fig.colorbar(im0, ax=axes[0], shrink=0.8, label="\u00b5T")

vmax = np.percentile(np.abs(res.b111_induced), 99)
im1 = axes[1].imshow(res.b111_induced, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].set_title("B\u2081\u2081\u2081 induced (\u00b5T)")
axes[1].set_xlabel("x (pixels)")
axes[1].set_ylabel("y (pixels)")
fig.colorbar(im1, ax=axes[1], shrink=0.8, label="\u00b5T")

plt.suptitle("B\u2081\u2081\u2081 from full ESR14N fit", y=1.01)
plt.tight_layout()
plt.show()

## 6. Parameter maps

`FitResult` stores all fitted parameters.  Use `res.parameters[name]` (shape
`(n_pol, n_frange, n_pixel)`) or `res.get_parameter_map(name)` for a 2-D image
(requires a 1-D slice first).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Center frequency — neg polarity, low range
center_map = res.parameters["center"][0, 0].reshape(res.scan_dimensions)
im0 = axes[0].imshow(center_map, cmap="plasma")
axes[0].set_title("Center frequency (GHz)\n[neg, low]")
fig.colorbar(im0, ax=axes[0], shrink=0.8)

# Contrast (first hyperfine dip) — neg polarity, low range
contrast_map = res.parameters["contrast_0"][0, 0].reshape(res.scan_dimensions)
im1 = axes[1].imshow(contrast_map, cmap="viridis")
axes[1].set_title("Contrast of dip 0 (a.u.)\n[neg, low]")
fig.colorbar(im1, ax=axes[1], shrink=0.8)

# Chi² fit quality — neg polarity, low range
chi2_map = res.parameters["chi2"][0, 0].reshape(res.scan_dimensions)
im2 = axes[2].imshow(chi2_map, cmap="hot_r",
                      vmax=np.percentile(chi2_map, 99))
axes[2].set_title("\u03c7\u00b2 (fit quality)\n[neg, low]")
fig.colorbar(im2, ax=axes[2], shrink=0.8)

for ax in axes:
    ax.set_xlabel("x (pixels)")
    ax.set_ylabel("y (pixels)")

plt.suptitle("Fitted parameter maps", y=1.01)
plt.tight_layout()
plt.show()

## Summary

| Step | API | What you get |
|------|-----|-------------|
| Load | `MatlabLoader` + `ODMRData.from_loader()` | 5-D `xr.DataArray` |
| Process | `ODMR` + processor pipeline | Binned, normalised `ODMRData` |
| Fit | `FitManager("ESR14N").fit(data, freq)` | `FitResult` |
| B₁₁₁ | `res.b111_remanent`, `res.b111_induced` | 2-D numpy arrays in µT |
| Parameters | `res.parameters[name]` | `(n_pol, n_frange, n_pixel)` arrays |

### Next steps
- See `load_mil2_fov1.ipynb` for a deeper walk-through including a **quick B₁₁₁ estimate
  from dip positions** that needs no GPU fitting.
- See `processor_tutorial.ipynb` for a dedicated look at the processing pipeline.
- See `tutorial_fitting.ipynb` for constraint management and advanced fitting.
- See `tutorial_models.ipynb` for model details (ESR14N / ESR15N / ESRSINGLE).